In [0]:
from pyspark.sql import functions as F

base_path = "/Volumes/voe_bem/bronze/raw_files"

def ingest_bronze(read_path, table_name, is_vra=False):
    if is_vra:
        # Lê como CSV sem header, com schema explícito (primeira linha "Atualizado em: ..." não tem separadores)
        from pyspark.sql.types import StructType, StructField, StringType
        vra_schema = StructType([
            StructField("c0", StringType(), True), StructField("c1", StringType(), True),
            StructField("c2", StringType(), True), StructField("c3", StringType(), True),
            StructField("c4", StringType(), True), StructField("c5", StringType(), True),
            StructField("c6", StringType(), True), StructField("c7", StringType(), True),
            StructField("c8", StringType(), True), StructField("c9", StringType(), True),
            StructField("c10", StringType(), True), StructField("c11", StringType(), True),
        ])
        raw = (
            spark.read
            .option("header", "false")
            .option("sep", ";")
            .schema(vra_schema)
            .csv(read_path)
        )
        # Descarta a linha "Atualizado em: ..." (só tem c0, sem separadores) e o cabeçalho repetido
        df = (
            raw
            .filter(F.col("c1").isNotNull())
            .filter(F.col("c0") != "ICAO Empresa Aérea")
            .toDF(
                "ICAO Empresa Aérea", "Número Voo", "Código Autorização (DI)",
                "Código Tipo Linha", "ICAO Aeródromo Origem", "ICAO Aeródromo Destino",
                "Partida Prevista", "Partida Real", "Chegada Prevista", "Chegada Real",
                "Situação Voo", "Código Justificativa"
            )
        )
        df = df.withColumn("_source_file", F.lit("VRA (múltiplos arquivos)"))
    else:
        df = (
            spark.read
            .option("header", "true")
            .option("inferSchema", "false")
            .option("sep", ";")
            .option("mode", "DROPMALFORMED")
            .csv(read_path)
            .withColumn("_source_file", F.col("_metadata.file_path"))
        )
    
    df = df.withColumn("_ingestion_timestamp", F.current_timestamp())
    
    invalid_chars = " ,;{}()\n\t="
    df = df.toDF(*[c.translate(str.maketrans(invalid_chars, "_" * len(invalid_chars))) for c in df.columns])
    
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)
    print(f"{table_name}: {df.count()} linhas gravadas")

# VRA - tratamento especial (is_vra=True)
ingest_bronze(f"{base_path}/VRA_*.csv", "voe_bem.bronze.vra", is_vra=True)

# Demais arquivos - sem alteração
ingest_bronze(f"{base_path}/AerodromosPublicos.csv", "voe_bem.bronze.aerodromos")
ingest_bronze(f"{base_path}/pda_empresas_aereas_nacionais.csv", "voe_bem.bronze.empresas_nacionais")
ingest_bronze(f"{base_path}/pda_empresas_aereas_estrangeiros.csv", "voe_bem.bronze.empresas_estrangeiras")